# REINFORCE vs. Pretraining: How Are Tokens Updated?

## 1. Minimal Example

**Prompt:** `2+2=?`

- **Pretraining target:** `abcdefg`
- **REINFORCE sampled completion:** `abcdef4`
- **Reward:** $R=1$, because the final answer is `4`.

Assume each character represents one token.

## 2. REINFORCE: Updating a Sampled Completion

The probability of generating `abcdef4` is:
$$
\begin{aligned}
P(abcdef4\mid x)
={}&P(a\mid x)\\
&\times P(b\mid xa)\\
&\times P(c\mid xab)\\
&\times\cdots\\
&\times P(4\mid xabcdef)
\end{aligned}
$$

a.k.a:

$$
P(y\mid x)=\prod_{t=1}^{7}P(y_t\mid x,y_{<t})
$$

Taking the logarithm converts the product into a sum:

$$
\log P(y\mid x)=\sum_{t=1}^{7}\log P(y_t\mid x,y_{<t})
$$

The REINFORCE gradient estimator is:

$$
\boxed{
g=R\sum_{t=1}^{7}\nabla_\theta\log P(y_t\mid x,y_{<t})
}
$$

Since $R=1$, every sampled token contributes to the gradient.

**Numerical example**

Assume a learning rate of $0.2$, with two possible next tokens at each position.

For the first token `a`:

$$
P(a\mid x)=0.5
$$

The log-probability gradient with respect to the two logits is:

$$
\nabla_z\log P(a)=[0.5,-0.5]
$$

A gradient-ascent update gives:

$$
z_{\text{new}}=[0,0]+0.2[0.5,-0.5]=[0.1,-0.1]
$$

After softmax:

$$
P(a\mid x):\quad 0.5\rightarrow 0.5498
$$

For the final token `4`, assume:

$$
P(4\mid xabcdef)=0.2,\qquad P(g\mid xabcdef)=0.8
$$

Then:

$$
\nabla_z\log P(4)=[0.8,-0.8]
$$

$$
\Delta z=0.2[0.8,-0.8]=[0.16,-0.16]
$$

After softmax:

$$
P(4\mid xabcdef):\quad 0.20\rightarrow 0.2561
$$

**Result:** REINFORCE reinforces the entire sampled sequence `abcdef4`, not just the correct final token `4`.

This numerical example assumes independent logits for each position. A real Transformer shares parameters across positions, so individual token probabilities are not guaranteed to increase after a joint parameter update.

## 3. Does the Prompt Receive Gradients?

**The prompt is used as context, but its tokens are not prediction targets in the REINFORCE loss.**

For:

- Prompt: `2+2=?`
- Completion: `abcdef4`

The loss is calculated over the seven completion tokens only.

However, gradients still flow through the neural network computations that process the prompt. Therefore, model parameters involved in interpreting the prompt can also be updated.

**Key distinction:**

- Prompt tokens: conditioning context, no direct token-prediction loss.
- Completion tokens: contribute directly to the policy-gradient objective.

## 4. Comparison with Pretraining

Suppose the training dataset provides `abcdefg` as the target completion.

With teacher forcing, the model maximizes:

$$
\log P(abcdefg\mid x)
=
\sum_{t=1}^{7}\log P(y_t^*\mid x,y_{<t}^*)
$$

The gradient-ascent direction for standard cross-entropy training is:

$$
g_{\text{PT}}=
\sum_{t=1}^{7}\nabla_\theta\log P(y_t^*\mid x,y_{<t}^*)
$$

REINFORCE instead uses a completion sampled from the current policy:

$$
g_{\text{RL}}=
R\sum_{t=1}^{7}\nabla_\theta\log P(y_t\mid x,y_{<t})
$$

**Key insight:** For a fixed completion with $R=1$, the REINFORCE sample gradient has the same mathematical form as a teacher-forced cross-entropy gradient on that completion (assuming the same loss scaling).

The fundamental difference is **where the completion comes from and what objective is being optimized**:

- Pretraining learns from dataset-provided targets.
- REINFORCE samples from the policy and uses rewards to optimize expected reward.

If $R=0$, vanilla REINFORCE produces zero gradient for that sample. With an advantage $A=R-b$, the gradient may instead be positive or negative.

## 5. The Credit Assignment Problem

REINFORCE receives:

`abcdef4` → Reward = 1

It does not know that only the final `4` was necessary to answer the question correctly.

Consequently, the same sequence-level reward multiplies the log-probability gradient of **every generated token**, including the irrelevant prefix `abcdef`.

This is the **credit assignment problem**: how can we identify which actions actually contributed to a successful outcome?

The reward multiplier is shared across tokens, but the individual token gradients are generally different.

## 6. Summary: Pretraining vs. REINFORCE

| | Pretraining / Teacher Forcing | REINFORCE |
|---|---|---|
| **Where does the completion come from?** | Dataset provides `abcdefg` | Policy samples `abcdef4` |
| **Which tokens are updated?** | Target tokens `a,b,c,d,e,f,g` | Sampled tokens `a,b,c,d,e,f,4` |
| **Is a reward required?** | No | Yes |
| **What is optimized?** | Increase the probability of dataset target tokens | Maximize expected reward |
| **Gradient weight for this sample** | 1 (standard cross-entropy) | $R$ (or advantage $A$) |